> **Provenance des données** — Les fichiers produits par `utils_louisfarm.py` sont synthétiques et reproductibles. Les noms de pays contextualisent les exercices ; les observations ne proviennent pas d’une enquête ni d’une institution financière réelle. Les résultats ne décrivent pas les populations de ces pays.
> Pour votre projet, documentez la source, la date, les unités et les droits d’utilisation. Les données personnelles doivent être anonymisées.


# LouisFarm — Semaine 7 : Classification & ML Avance
## Dataset : Microcredit Ghana (5000 clients, 12% de defauts)

**Objectif :** Construire un modele de scoring credit avec gestion des donnees desequilibrees.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme(style="whitegrid")
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                              roc_auc_score, roc_curve, f1_score)
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib, warnings; warnings.filterwarnings("ignore")
import sys; sys.path.insert(0, ".")
from utils_louisfarm import gen_microcredit_ghana

df = gen_microcredit_ghana(n=5000)
print(f"Dataset Microcredit Ghana: {df.shape}")
print(df.head(3))
print(f"\nTaux de defaut: {df.defaut_paiement.mean()*100:.1f}%  (DESEQUILIBRE IMPORTANT)")
print(df.defaut_paiement.value_counts())

## Lecon 7.1 — EDA Ciblee sur la Variable Cible

In [ ]:
# EDA CIBLE: Profil des defaillants vs non-defaillants
print("ANALYSE COMPARATIVE : Defaillants vs Non-defaillants")
print("=" * 55)

for col in ["age","revenu_mensuel_ghs","montant_credit_ghs","nbr_dependants"]:
    grp0 = df[df.defaut_paiement==0][col].mean()
    grp1 = df[df.defaut_paiement==1][col].mean()
    print(f"  {col:<30}: Sans defaut={grp0:>8.1f} | Avec defaut={grp1:>8.1f}")

print()
for col in ["genre","zone_rurale","historique_remboursement","garantie_type"]:
    taux = df.groupby(col)["defaut_paiement"].mean()*100
    print(f"  Taux defaut par {col}:")
    for cat, t in taux.sort_values(ascending=False).items():
        bar = "#" * int(t/2)
        print(f"    {str(cat):<20}: {t:>5.1f}% {bar}")
    print()

## Lecon 7.2 — Pipeline sklearn + Logistic Regression

In [ ]:
# PREPARATION
le_hist = LabelEncoder()
df["hist_enc"] = le_hist.fit_transform(df["historique_remboursement"])
df["ratio_mnt_rev"] = df["montant_credit_ghs"] / df["revenu_mensuel_ghs"].clip(1)

num_cols = ["age","revenu_mensuel_ghs","montant_credit_ghs","duree_mois",
            "nbr_dependants","zone_rurale","hist_enc","ratio_mnt_rev"]
cat_cols = ["genre","niveau_education","activite","garantie_type","region"]

X = df[num_cols + cat_cols]
y = df["defaut_paiement"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Defauts train: {y_train.mean()*100:.1f}%")

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

# Modele 1: Logistic Regression (baseline)
lr_pipe = Pipeline([("prep", prep), ("model", LogisticRegression(class_weight="balanced",max_iter=1000))])
lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)
lr_prob = lr_pipe.predict_proba(X_test)[:,1]

print("\n1. LOGISTIC REGRESSION (class_weight=balanced)")
print(classification_report(y_test, lr_pred, target_names=["Remboursement","Defaut"]))

## Lecon 7.3 — Random Forest + GradientBoosting + Oversampling

In [ ]:
# Modele 2: Random Forest avec oversampling
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X_train.copy(), y_train.copy())
print(f"Apres oversampling: {len(y_res)} echantillons | defauts: {y_res.mean()*100:.1f}%")

prep2 = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
rf_pipe = Pipeline([("prep", prep2), ("model", RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=42, n_jobs=-1))])
rf_pipe.fit(X_res, y_res)
rf_pred = rf_pipe.predict(X_test)
rf_prob = rf_pipe.predict_proba(X_test)[:,1]
print("\n2. RANDOM FOREST + OVERSAMPLING")
print(classification_report(y_test, rf_pred, target_names=["Remboursement","Defaut"]))

# Modele 3: Gradient Boosting
gb_pipe = Pipeline([("prep", ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])),
    ("model", GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.08, random_state=42))])
gb_pipe.fit(X_train, y_train)
gb_pred = gb_pipe.predict(X_test)
gb_prob = gb_pipe.predict_proba(X_test)[:,1]
print("\n3. GRADIENT BOOSTING")
print(classification_report(y_test, gb_pred, target_names=["Remboursement","Defaut"]))

## Lecon 7.4 — Evaluation : Matrices de Confusion et Courbes ROC

In [ ]:
# EVALUATION COMPAREE
from sklearn.metrics import recall_score, precision_score

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Semaine 7 - Evaluation modeles de classification (Credit Ghana)", fontweight="bold")

# Matrices de confusion
for ax, preds, name, color in [
    (axes[0], lr_pred, "LogReg", "Blues"),
    (axes[1], rf_pred, "Random Forest", "Greens"),
    (axes[2], gb_pred, "Gradient Boosting", "Oranges"),
]:
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Rembt","Defaut"])
    disp.plot(ax=ax, cmap=color, colorbar=False)
    r = recall_score(y_test, preds)
    p = precision_score(y_test, preds)
    ax.set_title(f"{name}\nRecall={r:.2%} | Precision={p:.2%}", fontsize=10)

plt.tight_layout()
plt.savefig("./s7_classification.png", dpi=100, bbox_inches="tight")
plt.show()

# Courbes ROC
fig2, ax = plt.subplots(figsize=(7, 5))
for probs, label, col in [(lr_prob,"LogReg","#C73E1D"),(rf_prob,"RF","#2E86AB"),(gb_prob,"GBM","#F18F01")]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, color=col, label=f"{label} (AUC={auc:.3f})")
ax.plot([0,1],[0,1],"k--", label="Hasard")
ax.set_xlabel("Taux faux positifs"); ax.set_ylabel("Taux vrais positifs")
ax.set_title("Courbes ROC — Scoring Credit Ghana")
ax.legend(); ax.grid(alpha=0.3)
plt.savefig("./s7_roc.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nCOMPARAISON FINALE:")
print(f"  LogReg: AUC={roc_auc_score(y_test,lr_prob):.3f} | Recall defaut={recall_score(y_test,lr_pred):.2%}")
print(f"  RF:     AUC={roc_auc_score(y_test,rf_prob):.3f} | Recall defaut={recall_score(y_test,rf_pred):.2%}")
print(f"  GBM:    AUC={roc_auc_score(y_test,gb_prob):.3f} | Recall defaut={recall_score(y_test,gb_pred):.2%}")

## Lecon 7.5 — Interpretation et Deploiement

In [ ]:
# IMPORTANCE DES FEATURES (RF)
rf_model = rf_pipe.named_steps["model"]
cat_feat_names = rf_pipe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(cat_cols)
feat_names = num_cols + list(cat_feat_names)
importances = pd.Series(rf_model.feature_importances_, index=feat_names).sort_values(ascending=False)

print("TOP 10 VARIABLES LES PLUS IMPORTANTES (Random Forest):")
print(importances.head(10).to_string())

# Sauvegarde
joblib.dump(gb_pipe, "./modele_scoring_ghana.pkl")

print("\n=== NOTE DE RECOMMANDATION POUR LA DIRECTION GhanaCredit ===")
print()
print("MODELE RETENU : Gradient Boosting (AUC=", round(roc_auc_score(y_test,gb_prob),3),")")
print()
print("INTERPRETATION DU SEUIL DE DECISION :")
print("  - Seuil 0.5 (standard) : Recall defaut ~ 60-65%")
print("  - Nous recommandons seuil 0.35 : Recall > 75%")
print("  - Rationale : le cout d un faux negatif (defaut manque)")
print("    est ~3x superieur au cout d un faux positif (bon client refuse)")
print()
print("VARIABLES CLES A SURVEILLER :")
print("  1. ratio_mnt_rev : Montant / Revenu (risque sur-endettement)")
print("  2. historique_remboursement : Indicateur le plus fiable")
print("  3. duree_mois : Engagements longs = risque accru")